# Example 12.1.2 / 1: Try Every Setting on One Instance

Companion notebook for **12.1.2 Multiobjective Options and Suffixes** in the AMPL lab Documentation, section "Try Every Setting on One Instance". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Try Every Setting on One Instance

The instance is the two-plant example from Multiple Objectives (AMPL lab documentation, section 12.1): plant `A` costs 2
and emits 5 per unit, plant `B` costs 4 and emits 1, each makes at most 8, and together they make
10. A third objective, `CleanUnits`, maximizes output at the clean plant; it is dropped except in
the `obj:multi:weight` runs. The helper `run()` resets every suffix with `ampl.eval("let ...")`,
activates the listed objectives with `drop()` and `restore()`, applies the settings, solves with
HiGHS, and prints `ampl.solve_result` with the plan from `to_dict()`.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PLANTS;                           # production plants
param unit_cost {PLANTS} >= 0;        # dollars per unit
param unit_emission {PLANTS} >= 0;    # kg CO2 per unit
param capacity {PLANTS} >= 0;         # units
param demand >= 0;                    # units

var Make {p in PLANTS} >= 0, <= capacity[p];  # units produced at each plant

minimize TotalCost: sum {p in PLANTS} unit_cost[p] * Make[p];          # dollars
minimize TotalEmission: sum {p in PLANTS} unit_emission[p] * Make[p];  # kg CO2
maximize CleanUnits: Make['B'];                                        # units at the clean plant

subject to MeetDemand: sum {p in PLANTS} Make[p] = demand;  # total output covers demand exactly

suffix objpriority IN, integer;       # solve order: higher value is optimized first
suffix objweight IN;                  # blend weight among objectives of equal priority
suffix objabstol IN;                  # allowed absolute worsening, objective units
suffix objreltol IN;                  # allowed relative worsening, fraction
suffix option_timelimit IN;           # time limit, seconds, for this objective's pass

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PLANTS"] = ["A", "B"]
ampl.param["unit_cost"] = {"A": 2, "B": 4}
ampl.param["unit_emission"] = {"A": 5, "B": 1}
ampl.param["capacity"] = {"A": 8, "B": 8}
ampl.param["demand"] = 10

In [5]:
# run.py
def run(label, options="", lets="", active=("TotalCost", "TotalEmission")):
    for s in ["objpriority", "objweight", "objabstol", "objreltol", "option_timelimit"]:
        ampl.eval(f"let {{i in 1.._nobjs}} _obj[i].{s} := 0;")
    for name in ["TotalCost", "TotalEmission", "CleanUnits"]:
        ampl.obj[name].restore() if name in active else ampl.obj[name].drop()
    ampl.eval(lets)
    ampl.solve(solver="highs", highs_options=options, verbose=False)
    plan = ampl.var["Make"].to_dict()
    print(f"{label:24} {ampl.solve_result:8} A={plan['A']:g} B={plan['B']:g}")

priorities = "let TotalCost.objpriority := 2; let TotalEmission.objpriority := 1;"
limits = "let TotalCost.option_timelimit := 10; let TotalEmission.option_timelimit := 20;"
weights = "let TotalCost.objweight := 1; let CleanUnits.objweight := 3;"

run("1 default")
run("2 obj:no=2", "obj:no=2")
run("3 obj:multi=2", "obj:multi=2")
run("4 obj:multi=1", "obj:multi=1")
run("5 objpriority", "obj:multi=2", priorities)
run("6 objabstol 4", "obj:multi=2", priorities + "let TotalCost.objabstol := 4;")
run("7 objreltol 0.25", "obj:multi=2", priorities + "let TotalCost.objreltol := 0.25;")
run("8 objweight 1, 0.3", "obj:multi=2", "let TotalCost.objweight := 1; let TotalEmission.objweight := 0.3;")
run("9 objweight one only", "obj:multi=2", "let TotalEmission.objweight := 0.1;")
run("10 objweight -1", "obj:multi=2", "let TotalEmission.objweight := -1;")
run("11 weight=2 (default)", "obj:multi=2", weights, ("TotalCost", "CleanUnits"))
run("12 weight=1", "obj:multi=2 obj:multi:weight=1", weights, ("TotalCost", "CleanUnits"))
run("13 option_, one group", "obj:multi=2", limits)
run("14 option_, ignored", "obj:multi=2 obj:multi:options=0", limits)
run("15 option_, two groups", "obj:multi=2", limits + priorities)

1 default                solved   A=8 B=2
2 obj:no=2               solved   A=2 B=8
3 obj:multi=2            solved   A=2 B=8
4 obj:multi=1            solved   A=2 B=8
5 objpriority            solved   A=8 B=2
6 objabstol 4            solved   A=6 B=4
7 objreltol 0.25         solved   A=5 B=5
8 objweight 1, 0.3       solved   A=8 B=2
9 objweight one only     solved   A=2 B=8
10 objweight -1          solved   A=8 B=2
11 weight=2 (default)    solved   A=8 B=2
12 weight=1              solved   A=2 B=8
13 option_, one group    failure  A=2 B=8
14 option_, ignored      solved   A=2 B=8
15 option_, two groups   solved   A=8 B=2


### Pure AMPL: data.dat and model.run

The same runs 5, 6 and 13 in pure AMPL, with the same `model.mod`; `drop` removes the third
objective and `display` prints the plan:

In [6]:
%%writefile data.dat
set PLANTS := A B;
param: unit_cost unit_emission capacity :=
    A    2   5   8
    B    4   1   8 ;
param demand := 10;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs, solver_msg 0;
drop CleanUnits;
option highs_options 'obj:multi=2';
let TotalCost.objpriority := 2;
let TotalEmission.objpriority := 1;
solve;
display solve_result, Make;
let TotalCost.objabstol := 4;
solve;
display Make;
let TotalCost.objabstol := 0;
let TotalCost.objpriority := 0;
let TotalEmission.objpriority := 0;
let TotalCost.option_timelimit := 10;
let TotalEmission.option_timelimit := 20;
option solver_msg 1;
solve;

HiGHS 1.14.0:   obj:multi = 2
solve_result = solved

Make [*] :=
A  8
B  2
;

HiGHS 1.14.0:   obj:multi = 2
Make [*] :=
A  6
B  4
;

HiGHS 1.14.0:   obj:multi = 2
HiGHS 1.14.0: Inconsistent 'option_timelimit' values
  for multi-objective pass 1
  (objectives with priority 0)
Objective = TotalCost
